# Notebook 18 — Hierarchical Semantic Boundary Distillation and adaptive constraints

**Purpose.** Test whether SABER's learning objective adds value beyond the
structured R-SBL mask.

The notebook compares three recovery objectives from the same physically pruned
starting point:

1. **CE+KD** — ordinary fine-type cross-entropy and knowledge distillation;
2. **HSBD-fixed** — binary, family, fine, directed-margin, KD, and Brier terms;
3. **HSBD-dual** — the same objective plus adaptive operational constraints.

This remains a validation-screening experiment. Multi-seed/test confirmation
belongs in Notebook 19 after the loss components pass their ablation gate.

In [ ]:
# Colab/repository bootstrap
from pathlib import Path
import os, sys, json, subprocess, platform

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

# Override with %env SABER_REPO=/your/path if your repository is elsewhere.
candidates = [
    os.environ.get("SABER_REPO"),
    "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression",
    str(Path.cwd()),
]
REPO = None
for candidate in candidates:
    if not candidate:
        continue
    p = Path(candidate).expanduser()
    if (p / "src").is_dir() and (p / "config").is_dir():
        REPO = p.resolve()
        break
if REPO is None:
    raise FileNotFoundError(
        "Repository not found. Set SABER_REPO or edit the candidate path."
    )
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("Repository:", REPO)
print("Python:", sys.version.split()[0], "| Platform:", platform.platform())

In [ ]:
# Install only the small SABER extension requirements.
# The original repository requirements must already be installed.
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-saber.txt"],
        check=True,
    )

In [ ]:
import yaml
from src.saber.adapters import SaberRepo

repo = SaberRepo.discover(REPO)
with open(REPO / "config" / "saber.yaml", "r", encoding="utf-8") as handle:
    SABER_CFG = yaml.safe_load(handle)

OUTPUT_ROOT = repo.output_root
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("SABER output root:", OUTPUT_ROOT)
print("Config branch:", SABER_CFG["project"]["branch"])

In [ ]:
from src.saber.bridge_ciciot import load_bridge
TRAIN_LOADER, VAL_LOADER, TEST_LOADER, MODEL, CLASS_NAMES = load_bridge()
print('bridge OK:', len(CLASS_NAMES), 'classes')

In [ ]:
# Repository bridge: auto-discovery first, explicit override second.
#
# If auto-discovery fails, set these objects using the same loader/model
# construction cells from the completed Computer Networks notebooks:
#   TRAIN_LOADER = ...
#   VAL_LOADER = ...
#   TEST_LOADER = ...
#   MODEL = ...
#   CLASS_NAMES = [...]
#
# MODEL must be the uncompressed CNN1D anchor and loaders must use the frozen
# train/validation/test split.

import torch
from src.saber.adapters import (
    auto_discover_loaders,
    auto_build_cnn,
    discover_anchor_checkpoint,
    infer_class_names_from_results,
    unpack_batch,
)

TRAIN_LOADER = globals().get("TRAIN_LOADER")
VAL_LOADER = globals().get("VAL_LOADER")
TEST_LOADER = globals().get("TEST_LOADER")
MODEL = globals().get("MODEL")
CLASS_NAMES = globals().get("CLASS_NAMES")

if any(obj is None for obj in (TRAIN_LOADER, VAL_LOADER, TEST_LOADER)):
    TRAIN_LOADER, VAL_LOADER, TEST_LOADER, _DATA_BUNDLE = auto_discover_loaders(repo)

first_batch = next(iter(VAL_LOADER))
x0, y0, env0 = unpack_batch(first_batch)
raw_example = x0[: min(8, len(x0))].float()

if CLASS_NAMES is None:
    CLASS_NAMES = infer_class_names_from_results(repo)

if MODEL is None:
    checkpoint = discover_anchor_checkpoint(repo)
    MODEL, MODEL_FACTORY_ERRORS = auto_build_cnn(
        n_features=int(x0.shape[-1]),
        n_classes=len(CLASS_NAMES),
        checkpoint=checkpoint,
    )
    print("Loaded checkpoint:", checkpoint)
    if MODEL_FACTORY_ERRORS:
        print("Model factory attempts that were skipped:", MODEL_FACTORY_ERRORS)

# Infer whether the historical CNN expects [B,F] and unsqueezes internally or
# expects an explicit [B,1,F] tensor. This decision is frozen for the notebook.
MODEL_INPUT_MODE = None
probe_out = None
candidate_inputs = [("raw", raw_example)]
if raw_example.ndim == 2:
    candidate_inputs.append(("unsqueeze_channel", raw_example.unsqueeze(1)))
errors = {}
MODEL.eval()
for mode, candidate in candidate_inputs:
    try:
        with torch.no_grad():
            probe_out = MODEL(candidate)
        MODEL_INPUT_MODE = mode
        EXAMPLE_INPUT = candidate
        break
    except Exception as exc:
        errors[mode] = repr(exc)

if MODEL_INPUT_MODE is None:
    raise RuntimeError(
        "Could not infer the CNN input convention. Set EXAMPLE_INPUT and "
        "MODEL_INPUT manually. Attempts: " + json.dumps(errors, indent=2)
    )

def MODEL_INPUT(x):
    if MODEL_INPUT_MODE == "unsqueeze_channel" and x.ndim == 2:
        return x.unsqueeze(1)
    return x

if probe_out.shape[1] != len(CLASS_NAMES):
    raise RuntimeError(
        f"Model outputs {probe_out.shape[1]} classes but CLASS_NAMES has "
        f"{len(CLASS_NAMES)} entries. Supply the exact label-encoder order."
    )

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL = MODEL.to(DEVICE)
EXAMPLE_INPUT = EXAMPLE_INPUT.to(DEVICE)
print("Device:", DEVICE)
print("Model:", type(MODEL).__name__)
print("Classes:", len(CLASS_NAMES))
print("Input convention:", MODEL_INPUT_MODE, tuple(EXAMPLE_INPUT.shape))

## 1. Select one R-SBL operating point and reconstruct its physical model

By default, choose the R-SBL budget with the lowest validation AWBIR among the
screened points. You can override `SELECTED_BUDGET` after reviewing Notebook 17.

In [ ]:
from copy import deepcopy
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch

from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.selectors import selection_to_prune_map
from src.saber.surgery import prune_cnn1d_channels
from src.saber.adapters import collect_logits, unpack_batch
from src.saber.metrics import (
    action_weighted_boundary_inversion_rate,
    full_model_audit,
)
from src.saber.losses import (
    HierarchyTensors,
    SaberLossConfig,
    edges_to_tensors,
    saber_loss,
)
from src.saber.constraints import (
    ConstraintTargets,
    DualRiskController,
    detached_metric_values,
    differentiable_risk_proxies,
    lagrangian_penalty,
)

OUT = OUTPUT_ROOT / "18_hierarchical_distillation"
OUT.mkdir(parents=True, exist_ok=True)
SELECT_DIR = OUTPUT_ROOT / "17_structured_selection"

taxonomy = ciciot2023_taxonomy(CLASS_NAMES)
robust_graph = pd.read_csv(
    OUTPUT_ROOT / "14_risk_graph" / "asvg_edges_robust.csv"
)

# SABER v2 operating point from the CALIBRATED screen: saber_v2 @ realized 40%
screen = pd.read_csv(SELECT_DIR / "structured_screening_results_calibrated.csv")
row = screen[(screen["method"] == "saber_v2") & (screen["budget"] == 0.40)].iloc[0]
SELECTED_BUDGET = float(row["budget"])
TAG = "saber_v2_r40cal"

removed = pd.read_csv(SELECT_DIR / f"{TAG}_removed_groups.csv")
prune_map = {
    str(layer): sorted(grp["channel_index"].astype(int).tolist())
    for layer, grp in removed.groupby("module_path")
}

# All three recovery objectives start from the IDENTICAL raw pruned anchor
# weights (no screening fine-tune), so differences are attributable to the
# recovery objective alone.
BASE_STUDENT, surgery_audit = prune_cnn1d_channels(
    MODEL,
    prune_map,
    EXAMPLE_INPUT.to(DEVICE),
    minimum_remaining_per_layer=int(
        SABER_CFG["groups"]["minimum_remaining_per_layer"]
    ),
)
BASE_STUDENT = BASE_STUDENT.to(DEVICE)

print("Selected method/budget:", row["method"], SELECTED_BUDGET,
      "| realized FLOP reduction:", float(row["flops_reduction_fraction"]))
display(surgery_audit)

## 2. Build hierarchy and directed-edge tensors

In [ ]:
hierarchy = HierarchyTensors(
    taxonomy.class_to_family_index,
    taxonomy.benign_index,
    len(taxonomy.families),
).to(DEVICE)
EDGE_SOURCE, EDGE_TARGET, EDGE_WEIGHT = edges_to_tensors(
    robust_graph, weight_column="robust_weight"
)
EDGE_SOURCE = EDGE_SOURCE.to(DEVICE)
EDGE_TARGET = EDGE_TARGET.to(DEVICE)
EDGE_WEIGHT = EDGE_WEIGHT.to(DEVICE)
print("Directed edges:", len(EDGE_SOURCE), "| Families:", taxonomy.families)

## 3. Derive teacher-relative constraint thresholds from validation data

The thresholds are not arbitrary absolute values. They are the teacher's
validation risk proxy plus the tolerances frozen in `config/saber.yaml`.

In [ ]:
from src.saber.losses import aggregate_family_probabilities

TEACHER = MODEL.to(DEVICE).eval()
VAL_TEACHER = np.load(
    OUTPUT_ROOT / "14_risk_graph" / "validation_teacher_outputs.npz",
    allow_pickle=True,
)
VAL_TEACHER_LOGITS = VAL_TEACHER["logits"]
VAL_LABELS = VAL_TEACHER["labels"].astype(np.int64)

def numpy_teacher_proxy_thresholds(logits, labels):
    z = torch.tensor(logits, dtype=torch.float32, device=DEVICE)
    y = torch.tensor(labels, dtype=torch.long, device=DEVICE)
    metrics = differentiable_risk_proxies(z, y, hierarchy)
    return detached_metric_values(metrics)

teacher_proxy = numpy_teacher_proxy_thresholds(VAL_TEACHER_LOGITS, VAL_LABELS)
tol = SABER_CFG["constraints"]
targets = ConstraintTargets(
    attack_miss_max=teacher_proxy["attack_miss"] + float(tol["attack_miss_tolerance"]),
    benign_false_alert_max=teacher_proxy["benign_false_alert"] + float(tol["benign_false_alert_tolerance"]),
    family_nll_max=teacher_proxy["family_nll"] + float(tol["family_nll_tolerance"]),
    class_cvar_loss_max=teacher_proxy["class_cvar"] + float(tol["class_cvar_tolerance"]),
)
print("Teacher proxy:", teacher_proxy)
print("Constraint targets:", targets.as_dict())

## 4. Shared validation and training functions

In [ ]:
from torch import nn

@torch.no_grad()
def validate_student(model):
    logits, labels, _ = collect_logits(
        model, VAL_LOADER, device=DEVICE, input_transform=MODEL_INPUT
    )
    audit = full_model_audit(logits, labels, taxonomy, DEFAULT_COST_PROFILES)
    awbir, _ = action_weighted_boundary_inversion_rate(
        VAL_TEACHER_LOGITS,
        logits,
        labels,
        robust_graph,
        weight_column="robust_weight",
    )
    audit["awbir"] = float(awbir)

    z = torch.tensor(logits, dtype=torch.float32, device=DEVICE)
    y = torch.tensor(labels, dtype=torch.long, device=DEVICE)
    proxy = detached_metric_values(
        differentiable_risk_proxies(z, y, hierarchy)
    )
    return audit, proxy

def train_variant(name, start_model, loss_config, *, use_duals, epochs=12):
    student = deepcopy(start_model).to(DEVICE)
    optimizer = torch.optim.Adam(
        student.parameters(),
        lr=float(SABER_CFG["distillation"]["learning_rate"]),
        weight_decay=float(SABER_CFG["distillation"]["weight_decay"]),
    )
    controller = DualRiskController(
        targets=targets,
        step_size=float(SABER_CFG["constraints"]["dual_step_size"]),
        max_dual=float(SABER_CFG["constraints"]["dual_max"]),
    )
    best_state = deepcopy(student.state_dict())
    best_score = -np.inf
    stale = 0
    history = []

    for epoch in range(1, epochs + 1):
        student.train()
        totals = {}
        seen = 0
        for batch in TRAIN_LOADER:
            x, y, _ = unpack_batch(batch)
            x = MODEL_INPUT(x.to(DEVICE))
            y = y.to(DEVICE).long()

            with torch.no_grad():
                teacher_logits = TEACHER(x)
            student_logits = student(x)

            proxy = differentiable_risk_proxies(
                student_logits, y, hierarchy
            )
            penalty = (
                lagrangian_penalty(proxy, controller)
                if use_duals
                else student_logits.new_zeros(())
            )
            loss, parts = saber_loss(
                student_logits,
                teacher_logits,
                y,
                hierarchy,
                EDGE_SOURCE,
                EDGE_TARGET,
                EDGE_WEIGHT,
                loss_config,
                extra_penalty=penalty,
            )
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            for key, value in parts.items():
                totals[key] = totals.get(key, 0.0) + float(value.detach().cpu()) * len(y)
            seen += len(y)

        audit, val_proxy = validate_student(student)
        if use_duals:
            controller.update(val_proxy)

        # Screening selection score rewards family fidelity while penalizing
        # directed inversions and balanced operational risk.
        selection_score = (
            audit["family_macro_f1"]
            - audit["awbir"]
            - audit["hsr_balanced_soc"]
        )
        row = {
            "variant": name,
            "epoch": epoch,
            **{f"train_{k}": v / max(seen, 1) for k, v in totals.items()},
            **{f"val_{k}": v for k, v in audit.items()},
            **{f"proxy_{k}": v for k, v in val_proxy.items()},
            **{f"dual_{k}": v for k, v in controller.lambdas.items()},
            "selection_score": selection_score,
        }
        history.append(row)

        if selection_score > best_score + 1e-6:
            best_score = selection_score
            best_state = deepcopy(student.state_dict())
            stale = 0
        else:
            stale += 1
        if stale >= int(SABER_CFG["distillation"]["patience"]):
            break

    student.load_state_dict(best_state)
    final_audit, final_proxy = validate_student(student)
    return student, pd.DataFrame(history), final_audit, final_proxy, controller

## 5. Define the three loss variants

Only the terms required by each comparison are enabled. This ensures the
ablation is interpretable rather than comparing unrelated training budgets.

In [ ]:
base_cfg = SABER_CFG["distillation"]

VARIANTS = {
    "ce_kd": SaberLossConfig(
        fine_weight=1.0,
        family_weight=0.0,
        binary_weight=0.0,
        margin_weight=0.0,
        kd_weight=float(base_cfg["kd_weight"]),
        calibration_weight=0.0,
        kd_temperature=float(base_cfg["kd_temperature"]),
    ),
    "hsbd_fixed": SaberLossConfig(
        fine_weight=float(base_cfg["fine_weight"]),
        family_weight=float(base_cfg["family_weight"]),
        binary_weight=float(base_cfg["binary_weight"]),
        margin_weight=float(base_cfg["margin_weight"]),
        kd_weight=float(base_cfg["kd_weight"]),
        calibration_weight=float(base_cfg["calibration_weight"]),
        kd_temperature=float(base_cfg["kd_temperature"]),
        margin_delta=float(base_cfg["margin_delta"]),
    ),
    "hsbd_dual": SaberLossConfig(
        fine_weight=float(base_cfg["fine_weight"]),
        family_weight=float(base_cfg["family_weight"]),
        binary_weight=float(base_cfg["binary_weight"]),
        margin_weight=float(base_cfg["margin_weight"]),
        kd_weight=float(base_cfg["kd_weight"]),
        calibration_weight=float(base_cfg["calibration_weight"]),
        kd_temperature=float(base_cfg["kd_temperature"]),
        margin_delta=float(base_cfg["margin_delta"]),
    ),
}
VARIANTS

## 6. Train and evaluate each variant

The loop writes one checkpoint and history per variant. Re-running a completed
variant is skipped unless `FORCE_RERUN=True`.

In [ ]:
FORCE_RERUN = False
summary_rows = []

for name, cfg in VARIANTS.items():
    ckpt_path = OUT / f"{TAG}_{name}.pt"
    history_path = OUT / f"{TAG}_{name}_history.csv"
    summary_path = OUT / f"{TAG}_{name}_summary.json"

    if ckpt_path.exists() and summary_path.exists() and not FORCE_RERUN:
        summary = json.loads(summary_path.read_text())
        summary_rows.append(summary)
        print("Skipped completed:", name)
        continue

    student, history, audit, proxy, controller = train_variant(
        name,
        BASE_STUDENT,
        cfg,
        use_duals=(name == "hsbd_dual"),
        epochs=int(SABER_CFG["distillation"]["epochs"]),
    )
    torch.save(
        {
            "state_dict": student.cpu().state_dict(),
            "variant": name,
            "budget": SELECTED_BUDGET,
            "loss_config": vars(cfg),
            "dual_state": controller.state_dict(),
            "class_names": CLASS_NAMES,
        },
        ckpt_path,
    )
    student = student.to(DEVICE)
    history.to_csv(history_path, index=False)
    summary = {
        "variant": name,
        "budget": SELECTED_BUDGET,
        **{k: float(v) for k, v in audit.items()},
        **{f"proxy_{k}": float(v) for k, v in proxy.items()},
        **{f"dual_{k}": float(v) for k, v in controller.lambdas.items()},
        "epochs_run": int(len(history)),
    }
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    summary_rows.append(summary)
    print(name, summary["fine_macro_f1"], summary["family_macro_f1"], summary["awbir"])

summary_table = pd.DataFrame(summary_rows).sort_values("variant")
summary_table.to_csv(OUT / "hsbd_variant_summary.csv", index=False)
display(summary_table[[
    "variant", "fine_macro_f1", "family_macro_f1", "awbir",
    "hsr_balanced_soc", "attack_to_benign_rate", "benign_to_attack_rate",
    "ece15", "epochs_run"
]])

## 7. Component gate and training traces

HSBD should improve at least two primary semantic outcomes beyond CE+KD, and the
dual controller should reduce validation constraint violations rather than only
increase training loss.

In [ ]:
primary = ["awbir", "hsr_balanced_soc", "family_macro_f1", "benign_to_attack_rate"]
indexed = summary_table.set_index("variant")
gate_details = {}

if {"ce_kd", "hsbd_fixed"}.issubset(indexed.index):
    wins_fixed = {
        "awbir": indexed.loc["hsbd_fixed", "awbir"] < indexed.loc["ce_kd", "awbir"],
        "hsr": indexed.loc["hsbd_fixed", "hsr_balanced_soc"] < indexed.loc["ce_kd", "hsr_balanced_soc"],
        "family_f1": indexed.loc["hsbd_fixed", "family_macro_f1"] > indexed.loc["ce_kd", "family_macro_f1"],
        "benign_false_alert": indexed.loc["hsbd_fixed", "benign_to_attack_rate"] < indexed.loc["ce_kd", "benign_to_attack_rate"],
    }
    gate_details["hsbd_fixed_vs_ce_kd"] = wins_fixed
else:
    wins_fixed = {}

if {"hsbd_fixed", "hsbd_dual"}.issubset(indexed.index):
    violations = {}
    for metric, threshold_name in [
        ("proxy_attack_miss", "attack_miss_max"),
        ("proxy_benign_false_alert", "benign_false_alert_max"),
        ("proxy_family_nll", "family_nll_max"),
        ("proxy_class_cvar", "class_cvar_loss_max"),
    ]:
        threshold = getattr(targets, threshold_name)
        violations[metric] = {
            "fixed": max(0.0, indexed.loc["hsbd_fixed", metric] - threshold),
            "dual": max(0.0, indexed.loc["hsbd_dual", metric] - threshold),
        }
    dual_improved = sum(v["dual"] < v["fixed"] for v in violations.values())
    gate_details["dual_constraint_violations"] = violations
else:
    dual_improved = 0

component_wins = sum(bool(v) for v in wins_fixed.values())
gate_pass = component_wins >= 2 and dual_improved >= 1
gate = {
    "gate": "G3_hierarchical_recovery_screen",
    "passed": bool(gate_pass),
    "hsbd_primary_wins": int(component_wins),
    "dual_constraints_improved": int(dual_improved),
    "details": gate_details,
}
(OUT / "G3_recovery_gate.json").write_text(json.dumps(gate, indent=2), encoding="utf-8")
gate

In [ ]:
import json
import numpy as np

def to_py(o):
    if isinstance(o, dict):
        return {k: to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_py(v) for v in o]
    if isinstance(o, np.generic):
        return o.item()
    return o

(OUT / "G3_recovery_gate.json").write_text(json.dumps(to_py(gate), indent=2), encoding="utf-8")
print(json.dumps(to_py(gate), indent=2))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name in VARIANTS:
    path = OUT / f"{TAG}_{name}_history.csv"
    if not path.exists():
        continue
    hist = pd.read_csv(path)
    axes[0].plot(hist["epoch"], hist["val_awbir"], marker="o", label=name)
    axes[1].plot(hist["epoch"], hist["val_family_macro_f1"], marker="o", label=name)
    axes[2].plot(hist["epoch"], hist["val_hsr_balanced_soc"], marker="o", label=name)
axes[0].set_title("Validation AWBIR")
axes[1].set_title("Validation family macro-F1")
axes[2].set_title("Validation balanced-SOC HSR")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.legend()
fig.tight_layout()
fig.savefig(OUT / "hsbd_training_traces.png", dpi=250)
plt.show()

## 8. Freeze the Stage-C artefacts

If G3 does not pass, inspect margin-loss scaling and cost profiles before adding
datasets or seeds. The correct response to a failed gate is redesign, not a
larger experiment.

In [ ]:
import subprocess
from src.saber.adapters import save_run_manifest

try:
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
except Exception:
    git_commit = None

artefacts = [
    OUT / "hsbd_variant_summary.csv",
    OUT / "G3_recovery_gate.json",
    OUT / "hsbd_training_traces.png",
]
save_run_manifest(
    OUT / "manifest.json",
    notebook="18_saber_hierarchical_distillation.ipynb",
    config=SABER_CFG,
    artifacts=artefacts,
    git_commit=git_commit,
)
print("Notebook 18 complete. G3 provisional pass:", gate_pass)